# Wiki Gap Detector
### 6-Layer LLM Pipeline — Google Colab

Run each cell **in order** from top to bottom.

**Layer 2** combines **cosine similarity** (structural disconnection signal) with **Claude** (strict knowledge-gap analyst prompt).

## Cell 1 — Install Dependencies

In [ ]:
!pip install anthropic pydantic pydantic-settings networkx pyyaml --quiet


## Cell 2 — API Key

In [ ]:
import os

os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # <- paste your key here


## Cell 3 — Models (Document + Gap)

In [ ]:
from __future__ import annotations

import uuid
from datetime import datetime
from enum import Enum
from typing import Any, Dict, List, Optional

from pydantic import BaseModel, Field


class DocumentType(str, Enum):
    MEETING_NOTES = "meeting_notes"
    DECISION_RECORD = "decision_record"
    PRESENTATION_DECK = "presentation_deck"
    PROJECT_PLAN = "project_plan"
    EVENT_PLAN = "event_plan"
    UNKNOWN = "unknown"


DOCUMENT_TEMPLATES: Dict[DocumentType, List[str]] = {
    DocumentType.MEETING_NOTES: [
        "date", "participants", "agenda", "discussion",
        "decisions", "action items", "next steps",
    ],
    DocumentType.DECISION_RECORD: [
        "date", "decision", "reason", "impact", "owner", "next steps",
    ],
    DocumentType.PRESENTATION_DECK: [
        "title", "date", "presenter", "purpose",
        "key points", "summary", "next steps",
    ],
    DocumentType.PROJECT_PLAN: [
        "project name", "goal", "scope", "approach",
        "tasks", "key features", "results",
    ],
    DocumentType.EVENT_PLAN: [
        "event name", "date", "location", "objective",
        "target audience", "agenda",
    ],
}


class DocumentSection(BaseModel):
    name: str
    content: str
    word_count: int = 0
    is_empty: bool = False


class ParsedDocument(BaseModel):
    doc_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    title: str
    source_path: str
    doc_type: DocumentType = DocumentType.UNKNOWN
    frontmatter: Dict = Field(default_factory=dict)
    sections: List[DocumentSection] = Field(default_factory=list)
    raw_content: str = ""
    word_count: int = 0

    def get_section(self, name: str) -> Optional[DocumentSection]:
        name_lower = name.lower()
        for section in self.sections:
            if name_lower in section.name.lower() or section.name.lower() in name_lower:
                return section
        return None

    def section_names(self) -> List[str]:
        return [s.name for s in self.sections]

    def full_text(self) -> str:
        parts = [self.title]
        for s in self.sections:
            parts.append(s.name)
            parts.append(s.content)
        return "\n\n".join(parts)


class GapCategory(str, Enum):
    STRUCTURAL = "STRUCTURAL"
    EXPLICIT_EXPRESSION = "EXPLICIT_EXPRESSION"
    IMPLICIT_EXPRESSION = "IMPLICIT_EXPRESSION"
    SEMANTIC = "SEMANTIC"
    RELATIONAL = "RELATIONAL"


class PriorityLevel(str, Enum):
    HIGH = "High"
    MEDIUM = "Medium"
    LOW = "Low"


class Evidence(BaseModel):
    source_layer: str
    description: str
    raw_evidence: Optional[str] = None
    matched_keyword: Optional[str] = None
    missing_fields: Optional[List[str]] = None
    graph_nodes: Optional[List[str]] = None
    graph_edges: Optional[List[Dict[str, str]]] = None


class RawGap(BaseModel):
    gap_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    gap_category: GapCategory
    gap_type: str
    document_id: str
    document_title: str
    affected_document_section: Optional[str] = None
    description: str
    evidence: Evidence
    confidence: float = Field(ge=0.0, le=1.0)
    severity_estimate: float = Field(default=5.0, ge=0.0, le=10.0)
    created_at: datetime = Field(default_factory=datetime.utcnow)


class AggregatedGap(BaseModel):
    gap_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    gap_category: GapCategory
    gap_type: str
    document_id: str
    document_title: str
    affected_document_section: Optional[str] = None
    description: str
    evidence_sources: List[Evidence] = Field(default_factory=list)
    confidence: float = Field(ge=0.0, le=1.0)
    severity_estimate: float = Field(default=5.0, ge=0.0, le=10.0)
    frequency: int = 1
    affected_entities: List[str] = Field(default_factory=list)
    source_gap_ids: List[str] = Field(default_factory=list)


class RankedGap(BaseModel):
    gap_id: str
    gap_category: GapCategory
    gap_type: str
    document_id: str
    document_title: str
    affected_document_section: Optional[str] = None
    description: str
    evidence_sources: List[Evidence]
    severity: float = Field(ge=0.0, le=10.0)
    impact: float = Field(ge=0.0, le=10.0)
    confidence: float = Field(ge=0.0, le=1.0)
    frequency: int
    final_score: float
    risk_level: PriorityLevel
    priority_score: float
    affected_entities: List[str]
    root_cause: str
    recommendation: str
    llm_reasoning: Optional[str] = None
    ranked_at: datetime = Field(default_factory=datetime.utcnow)


class GapReport(BaseModel):
    report_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    job_id: str
    generated_at: datetime = Field(default_factory=datetime.utcnow)
    documents_analyzed: List[str]
    total_gaps: int
    high_risk_count: int
    medium_risk_count: int
    low_risk_count: int
    gaps: List[RankedGap]
    pipeline_metadata: Dict[str, Any] = Field(default_factory=dict)


print("Models loaded.")


## Cell 4 — Config (Scoring Weights & Risk Multipliers)

In [ ]:
# FinalScore = (0.45*Severity + 0.40*Impact + 0.15*FreqNorm) * risk_multiplier
# Confidence is excluded from the formula and reported separately.
# risk_multiplier is based on LLM-assigned priority: High=1.1, Medium=1.0, Low=0.9

class Settings:
    anthropic_api_key: str = os.environ.get("ANTHROPIC_API_KEY", "")
    llm_model: str = "claude-haiku-4-5-20251001"
    weight_severity: float = 0.45
    weight_impact: float = 0.40
    weight_frequency: float = 0.15
    risk_multiplier_high: float = 1.1
    risk_multiplier_medium: float = 1.0
    risk_multiplier_low: float = 0.9

settings = Settings()
print("Config loaded. Model:", settings.llm_model)


## Cell 5 — Document Parser (Markdown → ParsedDocument)

In [ ]:
import re
from pathlib import Path
import yaml

_TYPE_SIGNALS = {
    DocumentType.PROJECT_PLAN: [
        "key features", "approach", "expected results", "goal", "results",
        "project name", "scope", "tasks", "project plan",
    ],
    DocumentType.MEETING_NOTES: [
        "participants", "agenda", "action items", "attendees",
        "meeting notes", "next steps", "decisions",
    ],
    DocumentType.DECISION_RECORD: [
        "decision", "reason", "impact", "owner", "decision record", "adr",
    ],
    DocumentType.PRESENTATION_DECK: [
        "presenter", "key points", "summary", "presentation", "deck", "slides",
    ],
    DocumentType.EVENT_PLAN: [
        "event name", "venue", "target audience", "event plan", "location",
        "objective", "registration",
    ],
}

def _detect_type(title, section_names, body):
    combined = " ".join([title] + section_names + [body[:500]]).lower()
    scores = {dt: 0 for dt in DocumentType if dt != DocumentType.UNKNOWN}
    for doc_type, signals in _TYPE_SIGNALS.items():
        for signal in signals:
            if signal in combined:
                scores[doc_type] += 1
    best = max(scores, key=lambda k: scores[k])
    return best if scores[best] > 0 else DocumentType.UNKNOWN

_HEADING_RE = re.compile(r"^(#{1,3})\s+(.+)$", re.MULTILINE)
_FM_RE = re.compile(r"^---[ \t]*\n(.*?)\n---[ \t]*\n", re.DOTALL)
_BOLD_HEADING_RE = re.compile(r"^\*{1,2}([A-Z][^*\n]+?)\*{1,2}\s*$", re.MULTILINE)

def _make_section(name, content):
    cleaned = content.strip()
    return DocumentSection(
        name=name.strip(), content=cleaned,
        word_count=len(cleaned.split()) if cleaned else 0,
        is_empty=not bool(cleaned),
    )

def _parse_markdown_text(raw, title_fallback, source_path):
    frontmatter, body = {}, raw
    fm_match = _FM_RE.match(raw)
    if fm_match:
        try:
            frontmatter = yaml.safe_load(fm_match.group(1)) or {}
        except yaml.YAMLError:
            frontmatter = {}
        body = raw[fm_match.end():]
    title = frontmatter.get("title", "") or ""
    if not title:
        h1 = re.search(r"^#\s+(.+)$", body, re.MULTILINE)
        title = h1.group(1).strip() if h1 else title_fallback
    sections = []
    heading_matches = list(_HEADING_RE.finditer(body))
    if heading_matches:
        intro = body[:heading_matches[0].start()].strip()
        if intro:
            sections.append(_make_section("Introduction", intro))
        for i, m in enumerate(heading_matches):
            end = heading_matches[i+1].start() if i+1 < len(heading_matches) else len(body)
            sections.append(_make_section(m.group(2), body[m.end():end].strip()))
    else:
        parts = _BOLD_HEADING_RE.split(body)
        if parts[0].strip():
            sections.append(_make_section("Introduction", parts[0]))
        i = 1
        while i < len(parts) - 1:
            sections.append(_make_section(parts[i], parts[i+1] if i+1 < len(parts) else ""))
            i += 2
    if not sections:
        sections.append(_make_section("Content", body))
    section_names = [s.name for s in sections]
    return ParsedDocument(
        title=title, source_path=source_path,
        doc_type=_detect_type(title, section_names, body),
        frontmatter=frontmatter, sections=sections,
        raw_content=raw, word_count=len(body.split()),
    )

def parse_markdown_file(file_path):
    path = Path(file_path)
    return _parse_markdown_text(path.read_text(encoding="utf-8"), path.stem, str(file_path))

def parse_markdown_text(text, title="Untitled", source_path=""):
    return _parse_markdown_text(text, title, source_path)

print("Document parser loaded.")


## Cell 6 — LLM Client (Anthropic API + All Prompts)

In [ ]:
import json
import logging
from anthropic import AsyncAnthropic

logger = logging.getLogger(__name__)
_llm_client = None

_SYSTEM_PROMPT = (
    "You are a senior knowledge-quality analyst specialising in wiki documentation gap detection. "
    "Always respond with syntactically valid JSON when instructed; no prose outside the JSON."
)

def _get_llm_client():
    global _llm_client
    if _llm_client is None:
        _llm_client = AsyncAnthropic(api_key=settings.anthropic_api_key)
    return _llm_client

def _extract_json(text):
    text = text.strip()
    fenced = re.search(r"```(?:json)?\s*([\s\S]+?)```", text)
    if fenced:
        text = fenced.group(1).strip()
    return json.loads(text)

async def _llm_call(prompt, max_tokens=2048):
    client = _get_llm_client()
    response = await client.messages.create(
        model=settings.llm_model,
        max_tokens=max_tokens,
        system=_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text


# ── Layer 2 prompt ─────────────────────────────────────────────────────────────

async def analyze_semantic_gaps(title, section_name, content, cosine_context=""):
    """Strict analyst prompt. cosine_context (pre-computed similarity signal) is
    injected between the section header and content so the LLM can use structural
    disconnection information when reasoning about knowledge gaps."""
    if not content.strip():
        return []

    cosine_block = f"\n{cosine_context}\n" if cosine_context.strip() else ""

    prompt = f"""You are a senior knowledge-gap analyst for software project documentation.
Your task is NOT to identify writing improvements.
Your task is ONLY to identify critical knowledge gaps that could negatively affect:
- project risk
- operational execution
- decision making
- traceability
- knowledge retention

Document: {title}
Section: {section_name}
{cosine_block}
Content:
{content[:2500]}

STRICT RULES:
1. Report a gap ONLY if essential project knowledge is missing.
2. Do NOT report writing style issues.
3. Do NOT report formatting issues.
4. Do NOT report minor clarifications.
5. Do NOT report generic improvement suggestions.

A valid gap must have direct impact on:
- project risk
- operational importance
- decision relevance
- knowledge loss

Gap categories allowed:
IMPLICIT_EXPRESSION:
- missing assumptions
- missing preconditions
- missing dependencies
- missing context

SEMANTIC:
- ambiguous commitments
- unclear decisions

Return MAXIMUM one gap per category.

If no significant gap exists: []

Return JSON only:
[
  {{
    "gap_type": "IMPLICIT_EXPRESSION" | "SEMANTIC",
    "description": "<specific description of the missing knowledge>",
    "evidence": "<direct quote or paraphrase that exposes the gap>",
    "severity_estimate": <integer 1-10>,
    "affected_section": "{section_name}"
  }}
]"""

    try:
        raw = await _llm_call(prompt, max_tokens=1500)
        return _extract_json(raw)
    except Exception as exc:
        logger.warning("Semantic gap analysis failed for '%s / %s': %s", title, section_name, exc)
        return []


# ── Layer 3 prompt ─────────────────────────────────────────────────────────────

async def extract_entities_and_relationships(title, content):
    prompt = f"""Extract entities and directional relationships for knowledge-graph construction.

Document: {title}
Content:
{content[:3500]}

Return a JSON object with exactly two keys:
{{
  "entities": [
    {{
      "id": "<short unique id, e.g. e1>",
      "type": "requirement | decision | task | feature | concept | participant | goal",
      "label": "<concise entity name>",
      "description": "<one-sentence description>",
      "source_section": "<section heading where found>"
    }}
  ],
  "relationships": [
    {{
      "source": "<entity id>",
      "target": "<entity id>",
      "type": "depends_on | implements | justifies | references | derived_from | relates_to",
      "description": "<brief description of the relationship>"
    }}
  ]
}}

Extract only substantive entities. Return ONLY valid JSON."""
    try:
        raw = await _llm_call(prompt, max_tokens=3000)
        result = _extract_json(raw)
        if not isinstance(result, dict):
            return {"entities": [], "relationships": []}
        result.setdefault("entities", [])
        result.setdefault("relationships", [])
        return result
    except Exception as exc:
        logger.warning("Entity extraction failed for '%s': %s", title, exc)
        return {"entities": [], "relationships": []}


# ── Layer 5 prompt ─────────────────────────────────────────────────────────────

async def rank_gap(gap_description, category, evidence_summary, document_title, section):
    prompt = f"""Evaluate this knowledge gap and provide an actionable recommendation.

Gap Category: {category}
Document: {document_title}
Section: {section or "N/A"}
Description: {gap_description}
Evidence: {evidence_summary[:800]}

Return a JSON object:
{{
  "severity": <float 0-10, how critical this gap is to project success and knowledge loss>,
  "impact": <float 0-10, effect on business goals, decision-making, and operational execution>,
  "confidence": <float 0.0-1.0, your confidence in this assessment — standalone metric, not used in scoring>,
  "priority": "High" | "Medium" | "Low",
  "root_cause": "<one sentence root-cause analysis>",
  "recommendation": "<concrete, fix-oriented action>",
  "reasoning": "<two-sentence explanation of the priority assessment>"
}}

Return ONLY valid JSON."""
    try:
        raw = await _llm_call(prompt, max_tokens=1024)
        return _extract_json(raw)
    except Exception as exc:
        logger.warning("Ranking failed for gap in '%s': %s", document_title, exc)
        return {
            "severity": 5.0, "impact": 5.0, "confidence": 0.5,
            "priority": "Medium",
            "root_cause": "Unable to determine - LLM call failed.",
            "recommendation": "Review and complete this section.",
            "reasoning": "Default fallback due to LLM error.",
        }

print("LLM client loaded.")


## Cell 7 — Layer 1: Rule-Based Detection (Structural + Explicit Expression)

In [ ]:
_MIN_SECTION_WORDS = 10

def _section_present(required_name, section_names):
    r = required_name.lower()
    return any(r in a.lower() or a.lower() in r for a in section_names)

def _section_content_ok(required_name, doc):
    r = required_name.lower()
    for section in doc.sections:
        if r in section.name.lower() or section.name.lower() in r:
            return (section.word_count >= _MIN_SECTION_WORDS), section.name
    return False, None

def detect_structural_gaps(doc):
    gaps = []
    if doc.doc_type == DocumentType.UNKNOWN:
        gaps.append(RawGap(
            gap_category=GapCategory.STRUCTURAL, gap_type="Structural Gap",
            document_id=doc.doc_id, document_title=doc.title,
            affected_document_section=None,
            description="Document type could not be identified; no template validation possible.",
            evidence=Evidence(source_layer="rule_based",
                              description="Document type detection returned UNKNOWN",
                              missing_fields=["document_type"]),
            confidence=0.9, severity_estimate=4.0,
        ))
        return gaps
    required_sections = DOCUMENT_TEMPLATES.get(doc.doc_type, [])
    section_names = doc.section_names()
    missing, empty = [], []
    for req in required_sections:
        if not _section_present(req, section_names):
            missing.append(req)
        else:
            ok, found_name = _section_content_ok(req, doc)
            if not ok:
                empty.append(found_name or req)
    for req in missing:
        gaps.append(RawGap(
            gap_category=GapCategory.STRUCTURAL, gap_type="Structural Gap",
            document_id=doc.doc_id, document_title=doc.title,
            affected_document_section=None,
            description=f"Missing required '{req}' section in {doc.doc_type.value} document.",
            evidence=Evidence(source_layer="rule_based",
                              description=f"Required section '{req}' absent per {doc.doc_type.value} template",
                              missing_fields=[req]),
            confidence=1.0, severity_estimate=7.0,
        ))
    for sec_name in empty:
        gaps.append(RawGap(
            gap_category=GapCategory.STRUCTURAL, gap_type="Structural Gap",
            document_id=doc.doc_id, document_title=doc.title,
            affected_document_section=sec_name,
            description=f"Required section '{sec_name}' exists but has insufficient content (< {_MIN_SECTION_WORDS} words).",
            evidence=Evidence(source_layer="rule_based",
                              description="Section present but effectively empty",
                              missing_fields=[sec_name]),
            confidence=1.0, severity_estimate=5.0,
        ))
    return gaps

EXPLICIT_KEYWORDS = [
    "TBD", "TODO", "FIXME", "unknown", "not specified",
    "to be determined", "further work needed",
    "further research required", "pending decision",
]
_KEYWORD_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in EXPLICIT_KEYWORDS) + r")\b", re.IGNORECASE
)

def detect_explicit_expression_gaps(doc):
    gaps = []
    for section in doc.sections:
        for match in _KEYWORD_PATTERN.finditer(section.content):
            keyword = match.group(1)
            start = max(0, match.start() - 60)
            end = min(len(section.content), match.end() + 60)
            context = section.content[start:end].replace("\n", " ").strip()
            gaps.append(RawGap(
                gap_category=GapCategory.EXPLICIT_EXPRESSION, gap_type="Explicit Expression Gap",
                document_id=doc.doc_id, document_title=doc.title,
                affected_document_section=section.name,
                description=f"Explicit incompleteness marker '{keyword}' in section '{section.name}'.",
                evidence=Evidence(source_layer="rule_based",
                                  description=f"Keyword '{keyword}' detected",
                                  matched_keyword=keyword, raw_evidence=context),
                confidence=1.0, severity_estimate=6.0,
            ))
    return gaps

def layer1_run(doc):
    return detect_structural_gaps(doc) + detect_explicit_expression_gaps(doc)

print("Layer 1 loaded.")


## Cell 8 — Layer 2: Semantic Analysis (Cosine Similarity + LLM)

**Two-step approach per section:**
1. **Cosine similarity** — compares this section against all others to detect structural isolation. The result is injected into the LLM prompt as `cosine_context`.
2. **LLM (Claude)** — strict knowledge-gap analyst prompt that uses both the content and the cosine signal to identify only high-impact `IMPLICIT_EXPRESSION` / `SEMANTIC` gaps.

In [ ]:
import asyncio
import math
from collections import Counter

_MIN_WORDS_FOR_ANALYSIS = 20
_LOW_SIM_THRESHOLD  = 0.15   # below -> flag as potentially disconnected
_HIGH_SIM_THRESHOLD = 0.35   # above -> flag as semantically related

_CATEGORY_MAP = {
    "IMPLICIT_EXPRESSION": GapCategory.IMPLICIT_EXPRESSION,
    "SEMANTIC":            GapCategory.SEMANTIC,
}
_TYPE_MAP = {
    "IMPLICIT_EXPRESSION": "Implicit Expression Gap",
    "SEMANTIC":            "Semantic Gap",
}


# ── Cosine similarity (pure Python, no extra deps) ──────────────────────────

def _term_freq(text):
    tokens = re.findall(r"\b[a-z]{3,}\b", text.lower())
    return Counter(tokens)

def _cosine(freq_a, freq_b):
    if not freq_a or not freq_b:
        return 0.0
    shared = set(freq_a) & set(freq_b)
    dot = sum(freq_a[w] * freq_b[w] for w in shared)
    mag_a = math.sqrt(sum(v * v for v in freq_a.values()))
    mag_b = math.sqrt(sum(v * v for v in freq_b.values()))
    if mag_a == 0 or mag_b == 0:
        return 0.0
    return dot / (mag_a * mag_b)

def _build_cosine_context(doc, current_name, current_content):
    other = [s for s in doc.sections
             if s.name != current_name and s.word_count >= _MIN_WORDS_FOR_ANALYSIS]
    if not other:
        return ""
    current_freq = _term_freq(current_content)
    sims = [(s.name, round(_cosine(current_freq, _term_freq(s.content)), 3)) for s in other]
    sims.sort(key=lambda x: x[1])
    low  = [(n, s) for n, s in sims if s < _LOW_SIM_THRESHOLD]
    high = [(n, s) for n, s in reversed(sims) if s >= _HIGH_SIM_THRESHOLD]
    if not low and not high:
        return ""
    lines = ["Cosine Similarity Signal (this section vs. others):"]
    if low:
        lines.append("  Potentially disconnected (low similarity):")
        for name, sim in low[:3]:
            lines.append(f"    - vs '{name}': {sim:.3f}")
    if high:
        lines.append("  Semantically related (high similarity):")
        for name, sim in high[:3]:
            lines.append(f"    - vs '{name}': {sim:.3f}")
    return "\n".join(lines)


# ── Per-section analysis ────────────────────────────────────────────────────

async def _analyze_section(doc, section_name, content):
    cosine_context = _build_cosine_context(doc, section_name, content)
    raw_gaps = await analyze_semantic_gaps(
        title=doc.title,
        section_name=section_name,
        content=content,
        cosine_context=cosine_context,
    )
    gaps = []
    for item in raw_gaps:
        if not isinstance(item, dict):
            continue
        gap_type_str = item.get("gap_type", "SEMANTIC").upper()
        category = _CATEGORY_MAP.get(gap_type_str, GapCategory.SEMANTIC)
        gaps.append(RawGap(
            gap_category=category,
            gap_type=_TYPE_MAP.get(gap_type_str, "Semantic Gap"),
            document_id=doc.doc_id,
            document_title=doc.title,
            affected_document_section=item.get("affected_section", section_name),
            description=str(item.get("description", "Unspecified semantic gap")),
            evidence=Evidence(
                source_layer="semantic",
                description=f"LLM+cosine {gap_type_str.lower()} gap",
                raw_evidence=str(item.get("evidence", "")),
            ),
            confidence=0.75,
            severity_estimate=float(item.get("severity_estimate", 5.0)),
        ))
    return gaps

async def layer2_run(doc):
    tasks = [
        _analyze_section(doc, s.name, s.content)
        for s in doc.sections if s.word_count >= _MIN_WORDS_FOR_ANALYSIS
    ]
    if not tasks:
        return []
    results = await asyncio.gather(*tasks, return_exceptions=True)
    gaps = []
    for r in results:
        if isinstance(r, Exception):
            print(f"  [WARN] Semantic analysis task failed: {r}")
        else:
            gaps.extend(r)
    return gaps

print("Layer 2 loaded (cosine + LLM).")


## Cell 9 — Layer 3: Graph Analysis (LLM Entity Extraction + NetworkX)

In [ ]:
import networkx as nx

_JUSTIFICATION_EDGES = {"justifies", "derived_from"}
_NEEDS_JUSTIFICATION  = {"decision", "requirement"}
_NEEDS_DEPENDENCY     = {"feature", "task"}
_ANCHOR_TYPES         = {"goal", "requirement"}

def _build_graph(entities, relationships):
    G = nx.DiGraph()
    for e in entities:
        eid = e.get("id", "")
        if eid:
            G.add_node(eid, label=e.get("label", eid), entity_type=e.get("type", "concept"),
                       description=e.get("description", ""), source_section=e.get("source_section", ""))
    for r in relationships:
        src, tgt = r.get("source", ""), r.get("target", "")
        if src and tgt and G.has_node(src) and G.has_node(tgt):
            G.add_edge(src, tgt, rel_type=r.get("type", "relates_to"), description=r.get("description", ""))
    return G

def _lbl(G, nid):
    return G.nodes[nid].get("label", nid)

def _detect_orphans(G, doc):
    orphans = [n for n in G.nodes if G.degree(n) == 0]
    if not orphans:
        return []
    labels = [_lbl(G, n) for n in orphans]
    return [RawGap(
        gap_category=GapCategory.RELATIONAL, gap_type="Relational Gap",
        document_id=doc.doc_id, document_title=doc.title, affected_document_section=None,
        description=f"{len(orphans)} isolated entities have no relationships: {', '.join(labels[:5])}.",
        evidence=Evidence(source_layer="graph", description="Isolated nodes (zero-degree)",
                          graph_nodes=labels, graph_edges=[]),
        confidence=0.85, severity_estimate=5.5,
    )]

def _detect_missing_rationale(G, doc):
    gaps = []
    for nid in G.nodes:
        if G.nodes[nid].get("entity_type") not in _NEEDS_JUSTIFICATION:
            continue
        if not any(G[u][nid].get("rel_type") in _JUSTIFICATION_EDGES for u in G.predecessors(nid)):
            label = _lbl(G, nid)
            ntype = G.nodes[nid].get("entity_type", "")
            gaps.append(RawGap(
                gap_category=GapCategory.RELATIONAL, gap_type="Relational Gap",
                document_id=doc.doc_id, document_title=doc.title,
                affected_document_section=G.nodes[nid].get("source_section") or None,
                description=f"The {ntype} '{label}' has no justification or rationale linked to it.",
                evidence=Evidence(source_layer="graph", description=f"{ntype} node without rationale link",
                                  graph_nodes=[label], graph_edges=[]),
                confidence=0.8, severity_estimate=7.0,
            ))
    return gaps

def _detect_missing_dependencies(G, doc):
    gaps = []
    anchor_ids = {n for n in G.nodes if G.nodes[n].get("entity_type") in _ANCHOR_TYPES}
    for nid in G.nodes:
        if G.nodes[nid].get("entity_type") not in _NEEDS_DEPENDENCY:
            continue
        if not any(nx.has_path(G.to_undirected(), nid, a) for a in anchor_ids):
            label = _lbl(G, nid)
            ntype = G.nodes[nid].get("entity_type", "")
            gaps.append(RawGap(
                gap_category=GapCategory.RELATIONAL, gap_type="Relational Gap",
                document_id=doc.doc_id, document_title=doc.title,
                affected_document_section=G.nodes[nid].get("source_section") or None,
                description=f"The {ntype} '{label}' cannot be traced back to any requirement or goal.",
                evidence=Evidence(source_layer="graph",
                                  description=f"{ntype} not connected to any goal/requirement anchor",
                                  graph_nodes=[label], graph_edges=[]),
                confidence=0.75, severity_estimate=6.5,
            ))
    return gaps

def _detect_disconnected(G, doc):
    if G.number_of_nodes() < 3:
        return []
    components = list(nx.weakly_connected_components(G))
    if len(components) <= 1:
        return []
    all_labels = [_lbl(G, n) for comp in components for n in comp]
    return [RawGap(
        gap_category=GapCategory.RELATIONAL, gap_type="Relational Gap",
        document_id=doc.doc_id, document_title=doc.title, affected_document_section=None,
        description=f"Knowledge graph has {len(components)} disconnected components.",
        evidence=Evidence(source_layer="graph",
                          description=f"{len(components)} weakly connected components",
                          graph_nodes=all_labels, graph_edges=[]),
        confidence=0.85, severity_estimate=6.0,
    )]

async def layer3_run(doc):
    extraction = await extract_entities_and_relationships(doc.title, doc.full_text())
    entities = extraction.get("entities", [])
    relationships = extraction.get("relationships", [])
    if not entities:
        return []
    G = _build_graph(entities, relationships)
    gaps = []
    gaps.extend(_detect_orphans(G, doc))
    gaps.extend(_detect_missing_rationale(G, doc))
    gaps.extend(_detect_missing_dependencies(G, doc))
    gaps.extend(_detect_disconnected(G, doc))
    return gaps

print("Layer 3 loaded.")


## Cell 10 — Layer 4: Aggregation (Dedup + Merge)

In [ ]:
_SIMILARITY_THRESHOLD = 0.65

def _token_set(text):
    return set(re.findall(r"\w+", text.lower()))

def _jaccard(a, b):
    ta, tb = _token_set(a), _token_set(b)
    if not ta and not tb:
        return 1.0
    return len(ta & tb) / len(ta | tb)

def _is_duplicate(existing, candidate):
    if existing.document_id != candidate.document_id:
        return False
    a, b = existing.affected_document_section, candidate.affected_document_section
    if not (a is None and b is None) and (a is None or b is None or a.lower() != b.lower()):
        return False
    return _jaccard(existing.description, candidate.description) >= _SIMILARITY_THRESHOLD

def _merge_into(agg, raw):
    agg.evidence_sources.append(raw.evidence)
    agg.source_gap_ids.append(raw.gap_id)
    agg.frequency += 1
    agg.confidence = (agg.confidence + raw.confidence) / 2
    if raw.severity_estimate > agg.severity_estimate:
        agg.severity_estimate = raw.severity_estimate
    if raw.evidence.graph_nodes:
        for node in raw.evidence.graph_nodes:
            if node not in agg.affected_entities:
                agg.affected_entities.append(node)

def _new_aggregated(raw):
    return AggregatedGap(
        gap_category=raw.gap_category, gap_type=raw.gap_type,
        document_id=raw.document_id, document_title=raw.document_title,
        affected_document_section=raw.affected_document_section,
        description=raw.description, evidence_sources=[raw.evidence],
        confidence=raw.confidence, severity_estimate=raw.severity_estimate,
        frequency=1,
        affected_entities=list(raw.evidence.graph_nodes) if raw.evidence.graph_nodes else [],
        source_gap_ids=[raw.gap_id],
    )

def layer4_run(raw_gaps):
    aggregated = []
    for raw in raw_gaps:
        found = False
        for agg in aggregated:
            if _is_duplicate(agg, raw):
                _merge_into(agg, raw)
                found = True
                break
        if not found:
            aggregated.append(_new_aggregated(raw))
    return aggregated

print("Layer 4 loaded.")


## Cell 11 — Layer 5: Ranking (LLM Severity/Impact Scoring)

In [ ]:
# FinalScore = (0.45*Severity + 0.40*Impact + 0.15*FreqNorm) * risk_multiplier
# risk_multiplier is driven by the LLM-assigned priority (High=1.1, Med=1.0, Low=0.9)
# Confidence is reported separately — not used in the scoring formula.

_RISK_MULTIPLIER = {
    PriorityLevel.HIGH:   settings.risk_multiplier_high,
    PriorityLevel.MEDIUM: settings.risk_multiplier_medium,
    PriorityLevel.LOW:    settings.risk_multiplier_low,
}

def _compute_score(severity, impact, frequency, priority):
    freq_norm = min(frequency / 5.0, 1.0) * 10.0
    raw = (settings.weight_severity * severity
         + settings.weight_impact   * impact
         + settings.weight_frequency * freq_norm)
    return round(raw * _RISK_MULTIPLIER.get(priority, 1.0), 3)

def _evidence_summary(gap):
    parts = []
    for ev in gap.evidence_sources[:3]:
        if ev.raw_evidence:      parts.append(ev.raw_evidence[:200])
        elif ev.missing_fields:  parts.append("Missing: " + ", ".join(ev.missing_fields))
        elif ev.graph_nodes:     parts.append("Nodes: " + ", ".join(ev.graph_nodes[:5]))
        else:                    parts.append(ev.description)
    return " | ".join(parts)

async def _rank_single(gap):
    llm_result = await rank_gap(
        gap_description=gap.description,
        category=gap.gap_category.value,
        evidence_summary=_evidence_summary(gap),
        document_title=gap.document_title,
        section=gap.affected_document_section or "",
    )
    severity     = float(llm_result.get("severity",   gap.severity_estimate))
    impact       = float(llm_result.get("impact",     5.0))
    confidence   = float(llm_result.get("confidence", gap.confidence))
    priority_str = llm_result.get("priority", "Medium")
    # Parse LLM priority first — it drives the risk multiplier
    priority = (PriorityLevel(priority_str) if priority_str in ("High","Medium","Low")
                else PriorityLevel.MEDIUM)
    final_score = _compute_score(severity, impact, gap.frequency, priority)
    return RankedGap(
        gap_id=gap.gap_id, gap_category=gap.gap_category, gap_type=gap.gap_type,
        document_id=gap.document_id, document_title=gap.document_title,
        affected_document_section=gap.affected_document_section,
        description=gap.description, evidence_sources=gap.evidence_sources,
        severity=severity, impact=impact, confidence=confidence, frequency=gap.frequency,
        final_score=final_score, risk_level=priority, priority_score=final_score,
        affected_entities=gap.affected_entities,
        root_cause=llm_result.get("root_cause", ""),
        recommendation=llm_result.get("recommendation", ""),
        llm_reasoning=llm_result.get("reasoning", ""),
    )

async def layer5_run(aggregated_gaps):
    ranked = list(await asyncio.gather(*[_rank_single(g) for g in aggregated_gaps]))
    ranked.sort(key=lambda g: g.priority_score, reverse=True)
    return ranked

print("Layer 5 loaded.")


## Cell 12 — Layer 6: Report Generation

In [ ]:
from collections import Counter as _Counter

def layer6_run(ranked_gaps, documents, job_id):
    high   = sum(1 for g in ranked_gaps if g.risk_level == PriorityLevel.HIGH)
    medium = sum(1 for g in ranked_gaps if g.risk_level == PriorityLevel.MEDIUM)
    low    = sum(1 for g in ranked_gaps if g.risk_level == PriorityLevel.LOW)
    category_counts = _Counter(g.gap_category.value for g in ranked_gaps)
    metadata = {
        "documents_count": len(documents),
        "doc_types": [d.doc_type.value for d in documents],
        "gap_category_breakdown": dict(category_counts),
        "average_priority_score": (
            round(sum(g.priority_score for g in ranked_gaps) / len(ranked_gaps), 3)
            if ranked_gaps else 0.0
        ),
    }
    return GapReport(
        job_id=job_id, documents_analyzed=[d.title for d in documents],
        total_gaps=len(ranked_gaps), high_risk_count=high,
        medium_risk_count=medium, low_risk_count=low,
        gaps=ranked_gaps, pipeline_metadata=metadata,
    )

print("Layer 6 loaded.")


## Cell 13 — Pipeline Orchestrator (All 6 Layers)

In [ ]:
import time
from collections import defaultdict

async def run_pipeline(documents, job_id):
    t0 = time.perf_counter()

    # Layer 1 - Rule-Based
    layer1_gaps = []
    for doc in documents:
        layer1_gaps.extend(layer1_run(doc))
    print(f"  Layer 1: {len(layer1_gaps)} structural/explicit gaps")

    # Layers 2 + 3 - concurrent per document
    all_results = await asyncio.gather(
        *[layer2_run(doc) for doc in documents],
        *[layer3_run(doc) for doc in documents],
        return_exceptions=True,
    )
    n = len(documents)
    layer2_gaps, layer3_gaps = [], []
    for i, result in enumerate(all_results):
        if isinstance(result, Exception):
            print(f"  [ERROR] Layer {'2' if i < n else '3'} doc[{i % n}]: {result}")
        elif i < n:
            layer2_gaps.extend(result)
        else:
            layer3_gaps.extend(result)
    print(f"  Layer 2: {len(layer2_gaps)} semantic gaps  (cosine + LLM)")
    print(f"  Layer 3: {len(layer3_gaps)} relational gaps")

    # Layer 4 - Aggregation + cap at 10 per document
    all_raw = layer1_gaps + layer2_gaps + layer3_gaps
    aggregated = layer4_run(all_raw)
    print(f"  Layer 4: {len(aggregated)} aggregated (from {len(all_raw)} raw)")
    per_doc = defaultdict(list)
    for g in aggregated:
        per_doc[g.document_id].append(g)
    aggregated = [
        g for gaps in per_doc.values()
        for g in sorted(gaps, key=lambda x: x.severity_estimate, reverse=True)[:10]
    ]
    print(f"  Layer 4 (capped): {len(aggregated)} gaps")

    # Layer 5 - Ranking
    ranked = await layer5_run(aggregated)
    print(f"  Layer 5: {len(ranked)} ranked gaps")

    # Layer 6 - Report
    report = layer6_run(ranked, documents, job_id)
    elapsed = round(time.perf_counter() - t0, 2)
    report.pipeline_metadata["elapsed_seconds"] = elapsed
    report.pipeline_metadata["raw_gaps_detected"] = len(all_raw)
    return report

print("Pipeline loaded.")


## Cell 14 — Load Your Documents

**Option A** — upload `.md` files:
```python
from google.colab import files
uploaded = files.upload()
documents = [parse_markdown_text(text.decode('utf-8'), title=name)
             for name, text in uploaded.items()]
```

**Option B** — paste text directly (edit `DOC1` below).

In [ ]:
DOC1 = """
# My Project Plan

## Goal
We want to build an AI-powered tool to detect knowledge gaps in wiki documents.

## Approach
Use a multi-layer pipeline combining rule-based and LLM-based detection.

## Key Features
- Structural gap detection
- Semantic analysis using Claude
- Graph-based relational gap detection
- Automated ranking and recommendations

## Results
TBD - further research required.
"""

# Add more: DOC2 = """..."""

documents = [
    parse_markdown_text(DOC1, title="My Project Plan"),
    # parse_markdown_text(DOC2, title="Second Document"),
]

for doc in documents:
    print(f"  Loaded: {doc.title!r}  (type={doc.doc_type.value}, sections={len(doc.sections)})")


## Cell 15 — Run the Full 6-Layer Pipeline

In [ ]:
import json

job_id = str(uuid.uuid4())
print(f"\n=== Wiki Gap Detector ===")
print(f"Documents : {len(documents)}")
print(f"Job ID    : {job_id}\n")

report = await run_pipeline(documents, job_id)

print(f"\n{'-'*60}")
print(f"  Total gaps  : {report.total_gaps}")
print(f"  High-risk   : {report.high_risk_count}")
print(f"  Medium-risk : {report.medium_risk_count}")
print(f"  Low-risk    : {report.low_risk_count}")
print(f"  Avg score   : {report.pipeline_metadata.get('average_priority_score', '?')}  "
      f"(conf reported separately: {report.pipeline_metadata.get('average_confidence', '?')}) ")
print(f"  Elapsed     : {report.pipeline_metadata.get('elapsed_seconds', '?')}s")
print(f"{'-'*60}\n")

print("TOP GAPS BY PRIORITY SCORE")
print(f"{'-'*60}")
for i, gap in enumerate(report.gaps[:10], 1):
    print(
        f"  {i:>2}. [{gap.risk_level.value:6}] score={gap.priority_score:.2f}  conf={gap.confidence:.2f}  [{gap.gap_category.value}]\n"
        f"      Doc    : {gap.document_title}\n"
        f"      Section: {gap.affected_document_section or '(document-level)'}\n"
        f"      Issue  : {gap.description[:120]}\n"
        f"      Fix    : {gap.recommendation[:120]}\n"
    )

report_json = json.dumps(report.model_dump(mode="json"), indent=2, ensure_ascii=False)
with open("gap_report_output.json", "w") as f:
    f.write(report_json)
print("Report saved to gap_report_output.json")

# Uncomment to download:
# from google.colab import files
# files.download("gap_report_output.json")


---

## Evaluation

The following 7 cells evaluate pipeline output against human annotations.

| Cell | What it does |
|------|--------------|
| **16** | Install eval deps; auto-export `system_output.csv` from pipeline report |
| **17** | Generate virtual demo `human_annotations_DEMO.csv` (37 KickstartAI gaps) |
| **18** | Upload your real `human_annotations.csv` — or fall back to demo table |
| **19** | Per-category confusion matrix (TP / FP / FN / TN, Precision, Recall, F1) |
| **20** | Macro-F1 + `sklearn` classification report |
| **21** | Top-K Agreement (K = 4) |
| **22** | Export `detection_results.xlsx` |

> **Human annotation format** — CSV with three columns: `gap_id`, `gap_category`, `human_rank`
> Valid categories: `STRUCTURAL` | `EXPLICIT_EXPRESSION` | `IMPLICIT_EXPRESSION` | `RELATIONAL` | `SEMANTIC`


## Cell 16 — Install Evaluation Dependencies + Export System Output

In [ ]:
!pip install scikit-learn openpyxl --quiet

import json
import pandas as pd

# Read the pipeline report saved by Cell 15
with open("gap_report_output.json") as _f:
    _raw = json.load(_f)

# Support both a single report dict and a list of reports (batch run)
_all_gaps = []
if isinstance(_raw, list):
    for _rpt in _raw:
        _all_gaps.extend(_rpt.get("gaps", []))
else:
    _all_gaps = _raw.get("gaps", [])

# Assign global system rank by priority_score (highest first)
_sorted = sorted(_all_gaps, key=lambda g: g["priority_score"], reverse=True)
system = pd.DataFrame([
    {
        "gap_id":             g["gap_id"],
        "predicted_category": g["gap_category"],
        "system_rank":        rank,
        "document_title":     g.get("document_title", ""),
        "section":            g.get("affected_document_section", ""),
        "priority_score":     round(g.get("priority_score", 0.0), 3),
        "confidence":         round(g.get("confidence", 0.0), 3),
    }
    for rank, g in enumerate(_sorted, start=1)
])
system.to_csv("system_output.csv", index=False)
print(f"Exported system_output.csv  ({len(system)} gaps)")
print(system["predicted_category"].value_counts().to_string())
system.head(10)


## Cell 17 — Virtual Human Annotation Table (Demo)

Hardcoded for the 3 KickstartAI documents (37 gaps).
Contains **6 deliberate category disagreements** with the system to produce realistic (non-trivial) evaluation metrics.

| gap_id prefix | System category | Human category |
|---------------|-----------------|----------------|
| f31eb922 | RELATIONAL | **IMPLICIT_EXPRESSION** |
| 87bd8b43 | IMPLICIT_EXPRESSION | **SEMANTIC** |
| 9966e90e | IMPLICIT_EXPRESSION | **SEMANTIC** |
| 99293bf5 | RELATIONAL | **STRUCTURAL** |
| feaa76cf | RELATIONAL | **STRUCTURAL** |
| 48307ccb | RELATIONAL | **STRUCTURAL** |


In [ ]:
import pandas as pd

# Virtual human annotations — 37 KickstartAI gaps (latest run)
# Comments mark the 6 intentional disagreements with the system model.
HUMAN_DEMO = [
    # gap_id,                                  gap_category,           human_rank
    # -- Doc 1: Enhancing environmental crime investigations with AI --
    ("67e9340d-e525-49bc-9317-6e40c0b31156",  "STRUCTURAL",            9),
    ("12f715bc-dcc4-410c-b3aa-e833d1da81ad",  "IMPLICIT_EXPRESSION",   1),
    ("9a1e6a74-c7de-49eb-8a04-b0b05bf9f035",  "SEMANTIC",              4),
    ("87bd8b43-0dfa-4c4f-8d5d-389119f3b7f7",  "SEMANTIC",             13),  # sys: IMPLICIT_EXPRESSION
    ("ffd067c7-da90-4c41-81dd-6fca6e16aae9",  "SEMANTIC",             15),
    ("61c164a4-b403-4270-9562-ab47f5fbf4c9",  "IMPLICIT_EXPRESSION",  20),
    ("06d1cc44-80e5-4b3f-a891-76ab1407f656",  "SEMANTIC",             24),
    ("feaa76cf-e03f-40b2-b4a3-979e5c165b55",  "STRUCTURAL",           32),  # sys: RELATIONAL
    ("48307ccb-aee5-44b0-95df-4d9b3587e473",  "STRUCTURAL",           33),  # sys: RELATIONAL
    ("232568f9-9b27-4e05-bc8e-804ef0ae1be9",  "RELATIONAL",           34),
    ("a896d185-c9d1-4ae2-9455-0b92c154f172",  "RELATIONAL",           37),
    # -- Doc 2: Supporting financial health with a GenAI chatbot --
    ("34645e0d-aa48-4447-b5a4-21e60cac5ccd",  "STRUCTURAL",            3),
    ("6430d4a1-cc9d-4cff-877b-e03875190e72",  "IMPLICIT_EXPRESSION",   5),
    ("19d12c77-f0b0-4f40-a580-db19c17ed309",  "IMPLICIT_EXPRESSION",   7),
    ("f31eb922-86c3-4713-9e87-b580dbf4f23a",  "IMPLICIT_EXPRESSION",  11),  # sys: RELATIONAL
    ("05a7361a-debe-43bc-92c7-eba2287cbf5c",  "SEMANTIC",             10),
    ("43ba9544-f8c7-4e3d-8310-2046c6026541",  "SEMANTIC",             12),
    ("5adf4bb5-2c58-4161-84c0-0cb6b8431e51",  "SEMANTIC",             14),
    ("04a2e745-2b0f-479d-861a-891166036c65",  "IMPLICIT_EXPRESSION",  18),
    ("9966e90e-7736-4b58-8a3f-5cf9db233836",  "SEMANTIC",             17),  # sys: IMPLICIT_EXPRESSION
    ("2a26d8f0-d8e2-40de-a7e3-265cd3805143",  "IMPLICIT_EXPRESSION",  19),
    ("b9eed96a-5d99-4eb0-8700-3a7104ccbd14",  "RELATIONAL",           23),
    ("f7fbfcc0-2375-41e7-abd6-037822590a3c",  "SEMANTIC",             25),
    ("2a59db2b-a50a-4095-90c7-e951b31e5af3",  "SEMANTIC",             27),
    ("9ed33c06-8f3f-40fd-b8d6-0aa8b170b79e",  "RELATIONAL",           35),
    ("2506f12e-8ccc-4a99-8a8f-c4190c187a5f",  "RELATIONAL",           36),
    # -- Doc 3: Unattended object detection at train stations --
    ("c4e77431-3f15-4b23-a7fb-7b02dcc64667",  "IMPLICIT_EXPRESSION",   2),
    ("2b67939c-0359-40c4-8897-b55f5d4ab177",  "IMPLICIT_EXPRESSION",   6),
    ("f8980b62-27ad-4763-8766-78a2cf019b5b",  "STRUCTURAL",           16),
    ("e65eee59-f74d-4c96-bf27-9703677b625c",  "SEMANTIC",              8),
    ("583cad5e-94dc-40b0-b796-64b7892c856c",  "SEMANTIC",             21),
    ("f1890a85-15e8-42a0-97da-d62b66d98af6",  "IMPLICIT_EXPRESSION",  22),
    ("99293bf5-200f-4dc5-a93c-65d5f3acc8e9",  "STRUCTURAL",           26),  # sys: RELATIONAL
    ("cc5477ef-9fb0-476a-9602-b8f86f916105",  "SEMANTIC",             28),
    ("19ba7c0a-f5aa-4097-b196-eb9f27927b15",  "SEMANTIC",             29),
    ("51dc5853-3f96-423c-aa98-3b2e643cf47b",  "IMPLICIT_EXPRESSION",  30),
    ("d6fe7688-dd83-46c8-bf2f-603c71f68d34",  "STRUCTURAL",           31),
]

human_demo_df = pd.DataFrame(HUMAN_DEMO, columns=["gap_id", "gap_category", "human_rank"])
human_demo_df.to_csv("human_annotations_DEMO.csv", index=False)

# Also write an empty template for filling in manually
template_cols = {"gap_id": [], "gap_category": [], "human_rank": []}
pd.DataFrame(template_cols).to_csv("human_annotations_template.csv", index=False)

print("Saved: human_annotations_DEMO.csv")
print("Saved: human_annotations_template.csv  (fill in manually)")
print()
print("Virtual annotation distribution:")
print(human_demo_df["gap_category"].value_counts().to_string())
human_demo_df


## Cell 18 — Upload Real Human Annotations

Upload your completed `human_annotations.csv` (columns: `gap_id`, `gap_category`, `human_rank`).

If you skip this cell (or cancel the upload), cells 19–22 will use the **virtual demo table** from Cell 17.

In [ ]:
try:
    from google.colab import files as _colab_files
    print("Select your human_annotations.csv  (Cancel to use demo table)")
    _up = _colab_files.upload()
    if _up:
        _fname = list(_up.keys())[0]
        human = pd.read_csv(_fname)
        print(f"Loaded real annotations: {len(human)} rows")
    else:
        print("No file uploaded — using DEMO table from Cell 17")
        human = human_demo_df
except Exception:
    # Running outside Colab or upload cancelled
    human = human_demo_df
    print("Using DEMO table (outside Colab or upload cancelled)")

print()
print("Human annotation distribution:")
print(human["gap_category"].value_counts().to_string())
human.head()


## Cell 19 — Per-Category Detection Evaluation

Binary classification per gap category:
- **TP** — system and human both assign this category
- **FP** — system assigns this category, human does not
- **FN** — human assigns this category, system does not
- **TN** — neither assigns this category

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# Merge on gap_id (only evaluate gaps present in both tables)
eval_df = pd.merge(
    system[["gap_id", "predicted_category", "system_rank"]],
    human[["gap_id", "gap_category", "human_rank"]],
    on="gap_id"
)
print(f"Evaluation on {len(eval_df)} matched gap IDs "
      f"(system has {len(system)}, human has {len(human)})")

CATEGORIES = [
    "STRUCTURAL",
    "EXPLICIT_EXPRESSION",
    "IMPLICIT_EXPRESSION",
    "RELATIONAL",
    "SEMANTIC",
]

results = []
for cat in CATEGORIES:
    sys_bin   = (eval_df["predicted_category"] == cat).astype(int)
    human_bin = (eval_df["gap_category"]        == cat).astype(int)

    if human_bin.sum() == 0 and sys_bin.sum() == 0:
        continue  # skip categories not present in either

    tn, fp, fn, tp = confusion_matrix(human_bin, sys_bin, labels=[0, 1]).ravel()
    prec = precision_score(human_bin, sys_bin, zero_division=0)
    rec  = recall_score(human_bin, sys_bin, zero_division=0)
    f1   = f1_score(human_bin, sys_bin, zero_division=0)

    results.append({
        "category":  cat,
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "precision": round(prec, 3),
        "recall":    round(rec,  3),
        "f1":        round(f1,   3),
    })

results_df = pd.DataFrame(results)
print()
print(results_df.to_string(index=False))


## Cell 20 — Classification Evaluation (Macro-F1)

In [ ]:
from sklearn.metrics import f1_score, classification_report

macro_f1 = f1_score(
    eval_df["gap_category"],
    eval_df["predicted_category"],
    average="macro",
    zero_division=0,
)
print(f"Macro-F1: {round(macro_f1, 3)}")
print()
print(classification_report(
    eval_df["gap_category"],
    eval_df["predicted_category"],
    zero_division=0,
))


## Cell 21 — Top-K Agreement (K = 4)

Fraction of the top-K ranked gaps (by human priority) that are also in the system's top-K.

In [ ]:
K = 4

topk_human  = set(eval_df.sort_values("human_rank").head(K)["gap_id"])
topk_system = set(eval_df.sort_values("system_rank").head(K)["gap_id"])
intersection = topk_human & topk_system
agreement    = len(intersection) / K

print(f"Top-{K} Human  : {sorted(topk_human)}")
print(f"Top-{K} System : {sorted(topk_system)}")
print(f"Intersection  : {sorted(intersection)}")
print(f"Top-{K} Agreement = {len(intersection)}/{K} = {round(agreement, 3)}")


## Cell 22 — Export Results to Excel

In [ ]:
results_df.to_excel("detection_results.xlsx", index=False)
print("Saved detection_results.xlsx")

# Download in Colab
try:
    from google.colab import files as _colab_files
    _colab_files.download("detection_results.xlsx")
except Exception:
    pass
